In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
import pandas as pd
import numpy as np
import sys
!{sys.executable} -m pip install hmmlearn


[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: C:\Users\hp\AppData\Local\Programs\Python\Python314\python.exe -m pip install --upgrade pip


### Using Markov Model

In [3]:
match_df = pd.read_csv(r"C:\Users\hp\Documents\football-data\results.csv")

In [4]:
match_df.head()

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
0,1872-11-30,Scotland,England,0.0,0.0,Friendly,Glasgow,Scotland,False
1,1873-03-08,England,Scotland,4.0,2.0,Friendly,London,England,False
2,1874-03-07,Scotland,England,2.0,1.0,Friendly,Glasgow,Scotland,False
3,1875-03-06,England,Scotland,2.0,2.0,Friendly,London,England,False
4,1876-03-04,Scotland,England,3.0,0.0,Friendly,Glasgow,Scotland,False


In [5]:
match_df['date'] = pd.to_datetime(match_df['date'])
start_date = '2000-01-01'
df_filtered = match_df[match_df['date'] >= start_date]

In [6]:
df_filtered.head()

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
24062,2000-01-04,Egypt,Togo,2.0,1.0,Friendly,Aswan,Egypt,False
24063,2000-01-07,Tunisia,Togo,7.0,0.0,Friendly,Tunis,Tunisia,False
24064,2000-01-08,Trinidad and Tobago,Canada,0.0,0.0,Friendly,Port of Spain,Trinidad and Tobago,False
24065,2000-01-09,Burkina Faso,Gabon,1.0,1.0,Friendly,Ouagadougou,Burkina Faso,False
24066,2000-01-09,Guatemala,Armenia,1.0,1.0,Friendly,Los Angeles,United States,True


In [7]:
df_morocco = df_filtered[(df_filtered['home_team'].str.contains('Morocco')) | 
                 (df_filtered['away_team'].str.contains('Morocco'))]

In [8]:
df_morocco.head()

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
24076,2000-01-18,Morocco,Trinidad and Tobago,1.0,0.0,Friendly,El Jadida,Morocco,False
24093,2000-01-25,Morocco,Congo,1.0,0.0,African Cup of Nations,Lagos,Nigeria,True
24109,2000-01-29,Tunisia,Morocco,0.0,0.0,African Cup of Nations,Lagos,Nigeria,True
24129,2000-02-03,Nigeria,Morocco,2.0,0.0,African Cup of Nations,Lagos,Nigeria,False
24380,2000-04-09,Gambia,Morocco,0.0,1.0,FIFA World Cup qualification,Bakau,Gambia,False


Now we read in our input csv data, create a home dataframe, and an away dataframe. The dataframe content contains the following columns:
- `State` : Win, loss, tie
- `State_Next` : Win, loss, tie (the next games result after this current game)
- `Observation` : Home, Away (did the team play at home or away)


In [9]:
def determine_state_and_observation(row, team):
    # Calculate the difference in scores
    dif_score = row['home_score'] - row['away_score']

    # Initialize state and observation
    state = ''
    observation = ''

    # Determine state and observation based on scores and teams
    if dif_score == 0:
        state = 'T' # Tie
        observation = 'H' if row['home_team'] == team else 'A'
    elif dif_score > 0:
        if row['home_team'] == team:
            state = 'W' # Win
            observation = 'H'
        else:
            state = 'L' # Loss
            observation = 'A'
    else: # dif_score < 0
        if row['home_team'] == team:
            state = 'L' # Loss
            observation = 'H'
        else:
            state = 'W' # Win
            observation = 'A'

    return pd.Series({'State': state, 'Observation': observation})


In [10]:
df_morocco.head()

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
24076,2000-01-18,Morocco,Trinidad and Tobago,1.0,0.0,Friendly,El Jadida,Morocco,False
24093,2000-01-25,Morocco,Congo,1.0,0.0,African Cup of Nations,Lagos,Nigeria,True
24109,2000-01-29,Tunisia,Morocco,0.0,0.0,African Cup of Nations,Lagos,Nigeria,True
24129,2000-02-03,Nigeria,Morocco,2.0,0.0,African Cup of Nations,Lagos,Nigeria,False
24380,2000-04-09,Gambia,Morocco,0.0,1.0,FIFA World Cup qualification,Bakau,Gambia,False


In [11]:
df_morocco = df_filtered[(df_filtered['home_team'].str.contains('Morocco')) |
 (df_filtered['away_team'].str.contains('Morocco'))]
df_morocco = df_morocco.copy()

In [12]:
df_morocco.head()

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
24076,2000-01-18,Morocco,Trinidad and Tobago,1.0,0.0,Friendly,El Jadida,Morocco,False
24093,2000-01-25,Morocco,Congo,1.0,0.0,African Cup of Nations,Lagos,Nigeria,True
24109,2000-01-29,Tunisia,Morocco,0.0,0.0,African Cup of Nations,Lagos,Nigeria,True
24129,2000-02-03,Nigeria,Morocco,2.0,0.0,African Cup of Nations,Lagos,Nigeria,False
24380,2000-04-09,Gambia,Morocco,0.0,1.0,FIFA World Cup qualification,Bakau,Gambia,False


In [13]:
df_Scotland = df_filtered[(df_filtered['home_team'].str.contains('Scotland')) |
 (df_filtered['away_team'].str.contains('Scotland'))]
df_Scotland = df_Scotland.copy()


In [14]:
df_Scotland.head()

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
24323,2000-03-29,Scotland,France,0.0,2.0,Friendly,Glasgow,Scotland,False
24462,2000-04-26,Netherlands,Scotland,0.0,0.0,Friendly,Arnhem,Netherlands,False
24543,2000-05-30,Republic of Ireland,Scotland,1.0,2.0,Friendly,Dublin,Republic of Ireland,False
24861,2000-09-03,Latvia,Scotland,0.0,1.0,FIFA World Cup qualification,Riga,Latvia,False
24933,2000-10-07,San Marino,Scotland,0.0,2.0,FIFA World Cup qualification,Serravalle,San Marino,False


In [15]:
team = 'Morocco'
df_morocco.loc[:, ['State', 'Observation']] = df_morocco.apply(determine_state_and_observation, axis=1, args=(team,))

df_morocco.drop(columns=["date", "home_team", "away_team", "home_score", "away_score", "tournament", "city", "country", "neutral"], inplace=True)
# Bring up next row state to this row as "StateNext" which represents the HMM state transition
df_morocco['StateNext'] = df_morocco['State'].shift(-1)

In [16]:
team_ = 'Morocco'
df_Scotland.loc[:, ['State', 'Observation']] = df_Scotland.apply(determine_state_and_observation, axis=1, args=(team_,))
df_Scotland.drop(columns=["date", "home_team", "away_team", "home_score", "away_score", "tournament", "city", "country", "neutral"], inplace=True)
# Bring up next row state to this row as "StateNext" which represents the HMM state transition
df_Scotland['StateNext'] = df_Scotland['State'].shift(-1)

In [17]:
df_Scotland.head()

,State,Observation,StateNext
24323,W,A,T
24462,T,A,W
24543,W,A,W
24861,W,A,W
24933,W,A,T


In [18]:
home_df = df_morocco
away_df = df_Scotland

states = ['W', 'L', 'T']
observations = ['H', 'A']

In this cell we calculate the home team and away teams hidden state transition matrix. The hidden state transition matrix includes the following transitions:
```
  W L T
W x x x
L x x x
T x x x
```

Some of the code may look a little hacky, because some transitions don't exist in the data set, so we have to clean it up / explicitly set things to 0 if that's the case.

In [19]:
def get_transition_matrix(df, label):
  # Count total state values (total wins, total losses, total ties)
  state_series = df['State'].value_counts()
  for state in states:
    if state not in state_series:
      # If a team never won/lost/tied set the count to 0
      state_series[state] = 0
  print(f"{label}: Original total wins, losses, ties")
  print(state_series)
  print()

  start_probability = np.array([state_series['W'] / len(df), state_series['L'] / len(df), state_series['T'] / len(df)])
  print(f"{label}: Start probability wins, losses, ties")
  print(start_probability)
  print()

  # Subtract the last state value from the counts since there is no transition to go to for it
  last_result = df[-1:]["State"].values[0]
  state_series[last_result] = state_series[last_result] - 1
  print(f"{label}: Fixed total wins, losses, ties")
  print(state_series)
  print()

  # Get a matrix of all the State -> StateNext combinations
  matrix_states = df.groupby(['State', 'StateNext']).size()
  # Fix any missing transitions
  for state1 in states:
    if state1 not in matrix_states:
      # If a team never won/lost/tied just set it to the initial probability
      matrix_states.at[state1] = pd.Series()
      matrix_states.at[state1, 'W'] = 1/3
      matrix_states.at[state1, 'L'] = 1/3
      matrix_states.at[state1, 'T'] = 1/3
    for state2 in states:
      if state2 not in matrix_states[state1]:
        # If a team never went from one state to another (like Win --> Tie) set it to 0 probability 
        # (because the other states like Win --> Win and Win --> Loss will sum to 1)
        matrix_states.at[state1, state2] = 0
  print(f"{label}: Total number of transitions from one state to another (like Win --> Win, Win --> Loss, etc...)")
  print(matrix_states)
  print()

  # Stupid hack to prevent division by 0, only happens because a team might have never won/lost/tied a game
  divisor_w = 1 if state_series['W'] == 0 else state_series['W']
  divisor_l = 1 if state_series['L'] == 0 else state_series['L']
  divisor_t = 1 if state_series['T'] == 0 else state_series['T']
  transition_matrix = np.array([[matrix_states['W']['W'] / divisor_w, matrix_states['W']['L'] / divisor_w, matrix_states['W']['T'] / divisor_w],
                                  [matrix_states['L']['W'] / divisor_l, matrix_states['L']['L'] / divisor_l, matrix_states['L']['T'] / divisor_l],
                                  [matrix_states['T']['W'] / divisor_t, matrix_states['T']['L'] / divisor_t, matrix_states['T']['T'] / divisor_t]
                                  ])
  print(f"{label}: Transition matrix")
  print('Format')
  print('   W L T')
  print(' W')
  print(' L')
  print(' T')
  print(transition_matrix)
  print()

  return start_probability, transition_matrix

# home_start_probability = np.array([1/3, 1/3, 1/3])
home_start_probability, home_transition_matrix = get_transition_matrix(home_df, "Home")

# away_start_probability = np.array([1/3, 1/3, 1/3])
away_start_probability, away_transition_matrix = get_transition_matrix(away_df, "Away")

Home: Original total wins, losses, ties
State
W    171
T     79
L     60
Name: count, dtype: int64

Home: Start probability wins, losses, ties
[0.5516129  0.19354839 0.25483871]

Home: Fixed total wins, losses, ties
State
W    171
T     79
L     59
Name: count, dtype: int64

Home: Total number of transitions from one state to another (like Win --> Win, Win --> Loss, etc...)
State  StateNext
L      L            15
       T            12
       W            32
T      L            15
       T            20
       W            44
W      L            30
       T            47
       W            94
dtype: int64

Home: Transition matrix
Format
   W L T
 W
 L
 T
[[0.5497076  0.1754386  0.2748538 ]
 [0.54237288 0.25423729 0.20338983]
 [0.55696203 0.18987342 0.25316456]]

Away: Original total wins, losses, ties
State
L    111
W     83
T     50
Name: count, dtype: int64

Away: Start probability wins, losses, ties
[0.34016393 0.45491803 0.20491803]

Away: Fixed total wins, losses, ties
State
L   

In this cell we calculate the home team and away teams observation transition matrix. The observation transition matrix includes the following transitions:
```
  H A
W x x
L x x
T x x
```

Some of the code may look a little hacky, because some transitions don't exist in the data set, so we have to clean it up / explicitly set things to 0 if that's the case.

In [20]:
def get_observation_matrix(df, label):
  # Count total observation values (total home, away games)
  observation_series = df['Observation'].value_counts()
  # Count total state values (total wins, total losses, total ties)
  observation_state_series = df['State'].value_counts()
  for state in states:
    if state not in observation_state_series:
      # If a team never won/lost/tied set the count to 0
      observation_state_series[state] = 0
  print(f"{label}: Total home, away games")
  print(observation_series)
  print()
  print(f"{label}: Total wins, losses, ties")
  print(observation_state_series)
  print()

  # Get a matrix of all the State -> Observation combinations
  matrix_observations = df.groupby(['State', 'Observation']).size()
  # Fix any missing transitions
  for state in states:
    if state not in matrix_observations:
      # If a team never won/lost/tied just set it to the initial probability
      matrix_observations.at[state] = pandas.Series()
      matrix_observations.at[state, 'H'] = .5
      matrix_observations.at[state, 'A'] = .5
    for observation in observations:
      if observation not in matrix_observations[state]:
        # If a team never went from one state to another (like Win --> Home) set it to 0 probability
        # (because the other states like Win --> Away will sum to 1)
        matrix_observations.at[state, observation] = 0
  print(f"{label}: Total number of transitions from one state to observation (like Win --> Away etc...)")
  print(matrix_observations)
  print()

  # Stupid hack to prevent division by 0, only happens because a team might have never won/lost/tied a game
  divisor_w = 1 if observation_state_series['W'] == 0 else observation_state_series['W']
  divisor_l = 1 if observation_state_series['L'] == 0 else observation_state_series['L']
  divisor_t = 1 if observation_state_series['T'] == 0 else observation_state_series['T']
  observation_matrix = np.array([[matrix_observations['W']['H'] / divisor_w, matrix_observations['W']['A'] / divisor_w],
                                    [matrix_observations['L']['H'] / divisor_l, matrix_observations['L']['A'] / divisor_l],
                                    [matrix_observations['T']['H'] / divisor_t, matrix_observations['T']['A'] / divisor_t]
                                  ])
  print(f"{label}: Observation matrix")
  print('Format')
  print('   H A')
  print(' W')
  print(' L')
  print(' T')
  print(observation_matrix)
  print()

  return observation_matrix

home_observation_matrix = get_observation_matrix(home_df, 'Home')
away_observation_matrix = get_observation_matrix(away_df, 'Away')

Home: Total home, away games
Observation
H    196
A    114
Name: count, dtype: int64

Home: Total wins, losses, ties
State
W    171
T     79
L     60
Name: count, dtype: int64

Home: Total number of transitions from one state to observation (like Win --> Away etc...)
State  Observation
L      A               31
       H               29
T      A               42
       H               37
W      A               41
       H              130
dtype: int64

Home: Observation matrix
Format
   H A
 W
 L
 T
[[0.76023392 0.23976608]
 [0.48333333 0.51666667]
 [0.46835443 0.53164557]]

Away: Total home, away games
Observation
A    244
Name: count, dtype: int64

Away: Total wins, losses, ties
State
L    111
W     83
T     50
Name: count, dtype: int64

Away: Total number of transitions from one state to observation (like Win --> Away etc...)
State  Observation
L      A              111
T      A               50
W      A               83
       H                0
L      H                0
T      H  

In this cell we calculate the home team and away teams posterior probability for each hidden state, based on whether the next game they play is at home or away.

Once we obtain the posterior probability for each teams hidden states, take the max of the following:
- Sum home team `W` posterior probability with away team `L` posterior probability, divide by 2
- Sum away team `W` posterior probability with home team `L` posterior probability, divide by 2
- Sum home and away team `T` posterior probability, divide by 2

In [21]:
from hmmlearn import hmm

NEXT_GAME_HOME = np.array([[0]])
NEXT_GAME_AWAY = np.array([[1]])

def get_model(start_probability, transition_matrix, observation_matrix, next_game):
  model = hmm.CategoricalHMM(n_components=3, init_params='')
  model.startprob_ = start_probability
  model.transmat_ = transition_matrix
  model.emissionprob_ = observation_matrix
  predict_proba = model.predict_proba(next_game)
  predict = model.predict(next_game)
  decode = model.decode(next_game)
  return predict_proba, predict, decode

predict_proba_home, predict_home, decode_home = get_model(home_start_probability, home_transition_matrix, home_observation_matrix, NEXT_GAME_HOME)
predict_proba_away, predict_away, decode_away = get_model(away_start_probability, away_transition_matrix, away_observation_matrix, NEXT_GAME_AWAY)

print(f"Individual predictions...")
print(f"Home: win, loss, tie prediction probability:\n{predict_proba_home}")
print(f"Home prediction:\n{states[predict_home[0]]}")
print(f"Log probability with home prediction:\n{decode_home[0]}, {states[decode_home[1][0]]}")
print()
print(f"Away: win, loss, tie prediction probability:\n{predict_proba_away}")
print(f"Away prediction:\n{states[predict_away[0]]}")
print(f"Log probability with away prediction:\n{decode_away[0]}, {states[decode_away[1][0]]}")
print()
print()

home_win_away_loss_pct = (predict_proba_home[0][0] + predict_proba_away[0][1]) / 2
home_loss_away_win_pct = (predict_proba_home[0][1] + predict_proba_away[0][0]) / 2
tie_pct = (predict_proba_home[0][2] + predict_proba_away[0][2]) / 2

print(f"Final predictions...")
print(f"Home team win and away team loses:\n{home_win_away_loss_pct}")
print(f"Home team loses and away team wins:\n{home_loss_away_win_pct}")
print(f"Tie:\n{tie_pct}")

Individual predictions...
Home: win, loss, tie prediction probability:
[[0.66326531 0.14795918 0.18877551]]
Home prediction:
W
Log probability with home prediction:
-0.8690378470236095, W

Away: win, loss, tie prediction probability:
[[0.34016393 0.45491803 0.20491803]]
Away prediction:
L
Log probability with away prediction:
-0.7876380239808677, L


Final predictions...
Home team win and away team loses:
0.5590916694546672
Home team loses and away team wins:
0.24406155904984944
Tie:
0.19684677149548346


So Morocco will win with probability of 0.56

### Using Machine Learning Algorithms 

In [22]:
matches = pd.read_csv(r"C:\Users\hp\Documents\football-data\international_matches.csv")
rank = pd.read_csv(r"C:\Users\hp\Documents\football-data\fifa_ranking-2023-07-20.csv")

Let's take a loot at each dataframes by .head().


In [23]:
matches.head()

,date,home_team,away_team,home_team_continent,away_team_continent,home_team_fifa_rank,away_team_fifa_rank,home_team_total_fifa_points,away_team_total_fifa_points,home_team_score,...,shoot_out,home_team_result,home_team_goalkeeper_score,away_team_goalkeeper_score,home_team_mean_defense_score,home_team_mean_offense_score,home_team_mean_midfield_score,away_team_mean_defense_score,away_team_mean_offense_score,away_team_mean_midfield_score
0,1993-08-08,Bolivia,Uruguay,South America,South America,59,22,0,0,3,...,No,Win,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1993-08-08,Brazil,Mexico,South America,North America,8,14,0,0,1,...,No,Draw,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1993-08-08,Ecuador,Venezuela,South America,South America,35,94,0,0,5,...,No,Win,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1993-08-08,Guinea,Sierra Leone,Africa,Africa,65,86,0,0,1,...,No,Win,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1993-08-08,Paraguay,Argentina,South America,South America,67,5,0,0,1,...,No,Lose,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [24]:
rank.head()

,rank,country_full,country_abrv,total_points,previous_points,rank_change,confederation,rank_date
0,1,Germany,GER,57.0,0.0,0,UEFA,1992-12-31
1,96,Syria,SYR,11.0,0.0,0,AFC,1992-12-31
2,97,Burkina Faso,BFA,11.0,0.0,0,CAF,1992-12-31
3,99,Latvia,LVA,10.0,0.0,0,UEFA,1992-12-31
4,100,Burundi,BDI,10.0,0.0,0,CAF,1992-12-31


Some countries have different names. Let's standardize it.

In [25]:
matches = matches.replace({"IR Iran": "Iran", "Korea Republic" : "South Korea"})
rank = rank.replace({"IR Iran": "Iran", "Korea Republic" : "South Korea"})

Feature Extraction :

I extracted some features for predicting match result, similar to the approach here. I haven't tried alot of experiments, so these might be not the best features. I recommend to try your own ideas here.

For your information, is_stake indicates whether the match is Friendly or not. Some teams tend to not do their best on friendly matches, so is_stake handles these cases.

In [26]:
matches['rank_difference'] = matches['home_team_fifa_rank'] - matches['away_team_fifa_rank']
matches['average_rank'] = (matches['home_team_fifa_rank'] + matches['away_team_fifa_rank'])/2
matches['point_difference'] = matches['home_team_total_fifa_points'] - matches['away_team_total_fifa_points']
matches['is_stake'] = matches['tournament'] != 'Friendly'
matches['is_worldcup'] = 'FIFA World Cup' in matches['tournament']

matches['score_difference'] = matches['home_team_score'] - matches['away_team_score'] # Note that this feature is not used in training
matches['is_won'] = matches['score_difference'] > 0 # Take draw as lost

## Training

In [27]:
from sklearn import linear_model
from sklearn import ensemble
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, roc_curve, roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures

# machine learning
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC, LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import Perceptron
from sklearn.linear_model import SGDClassifier
from sklearn.tree import DecisionTreeClassifier

In [28]:
X, y = matches.loc[:,['average_rank', 'rank_difference', 'point_difference', 'is_stake', 'is_worldcup']], matches['is_won']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

Let's try different machine learning models. In this notebook, we'll try the following.

* Logistic Regression
* Support Vector Machines
* KNN
* Gaussian Naive Bayes
* Decision Tree
* Random Forest

In [29]:
# Logistic Regression

logreg = LogisticRegression()
logreg.fit(X_train, y_train)
lg_pred = logreg.predict(X_test)
acc_log = round(logreg.score(X_test, y_test) * 100, 2)
acc_log

68.38

In [30]:
# Support Vector Machines

svc = SVC()
svc.fit(X_train, y_train)
svm_pred = svc.predict(X_test)
acc_svc = round(svc.score(X_test, y_test) * 100, 2)
acc_svc

68.17

In [31]:
# KNN

knn = KNeighborsClassifier(n_neighbors = 3)
knn.fit(X_train, y_train)
Y_pred = knn.predict(X_test)
acc_knn = round(knn.score(X_test, y_test) * 100, 2)
acc_knn

62.88

In [32]:
# Gaussian Naive Bayes

gaussian = GaussianNB()
gaussian.fit(X_train, y_train)
gnb_pred = gaussian.predict(X_test)
acc_gaussian = round(gaussian.score(X_test, y_test) * 100, 2)
acc_gaussian

68.36

In [33]:
# Decision Tree

decision_tree = DecisionTreeClassifier()
decision_tree.fit(X_train, y_train)
Y_pred = decision_tree.predict(X_test)
acc_decision_tree = round(decision_tree.score(X_test, y_test) * 100, 2)
acc_decision_tree

59.1

In [34]:
# Random Forest

random_forest = RandomForestClassifier(n_estimators=100)
random_forest.fit(X_train, y_train)
Y_pred = random_forest.predict(X_test)
random_forest.score(X_train, y_train)
acc_random_forest = round(random_forest.score(X_test, y_test) * 100, 2)
acc_random_forest

63.7

In [35]:
models = pd.DataFrame({
    'Model': ['Support Vector Machines', 'KNN', 'Logistic Regression', 
              'Random Forest', 'Naive Bayes',
              'Decision Tree'],
    'Score': [acc_svc, acc_knn, acc_log, 
              acc_random_forest, acc_gaussian, acc_decision_tree]})
models.sort_values(by='Score', ascending=False)

,Model,Score
2,Logistic Regression,68.38
4,Naive Bayes,68.36
0,Support Vector Machines,68.17
3,Random Forest,63.70
1,KNN,62.88
5,Decision Tree,59.10


I'll use Logistic Regression for final prediction. Ensembling top 3 models may work better though.

In [36]:
model = logreg

In [37]:
teams_df = ['Morocco','Scotland']

In [38]:
# let's define a small margin when we safer to predict draw then win
margin = 0.05

# let's define the rankings at a recent time
rank = rank.loc[(rank['rank_date'] == rank['rank_date'].max()) & (rank['country_full'].isin(teams_df))]

teams_rankings = rank.set_index(['country_full'])
teams_rankings.head()

,rank,country_abrv,total_points,previous_points,rank_change,confederation,rank_date
country_full,,,,,,,
Scotland,30,SCO,1520.24,1520.24,0,UEFA,2023-07-20
Morocco,14,MAR,1655.50,1655.50,1,CAF,2023-07-20


In [40]:
home = 'Morocco'
away = 'Scotland'

row = pd.DataFrame(np.array([[np.nan, np.nan, np.nan, True, True]]), columns=X_train.columns)

home_rank = teams_rankings.loc[home, 'rank']
home_points = teams_rankings.loc[home, 'total_points']
opp_rank = teams_rankings.loc[away, 'rank']
opp_points = teams_rankings.loc[away, 'total_points']

row['average_rank'] = (home_rank + opp_rank) / 2
row['rank_difference'] = home_rank - opp_rank
row['point_difference'] = home_points - opp_points

# Model Output
home_win_prob = model.predict_proba(row)[:,1][0]

# Determining Win / Draw / Lose based on home_win_prob
points = 0
if home_win_prob <= 0.5 - margin:
    print("{} wins with {:.2f}".format(away, 1-home_win_prob))
if home_win_prob > 0.5 - margin:
    points = 1
if home_win_prob >= 0.5 + margin:
    points = 3
    print("{} wins with {:.2f}".format(home, home_win_prob))
if points == 1:
    print("Draw")

Draw


In [41]:
print(home_win_prob)

0.534367497790682
